# 🚀 Fine-Tuning Snippy: In-Browser Code Snippet & Tool Calling AI Agent with Gemma 3 (270M)

This notebook demonstrates how to fine-tune **Gemma 3 270M** (`unsloth/gemma-3-270m-it`) as **Snippy** — an in-browser code snippet generator agent. Snippy dynamically selects generic browser tools (`set_background_color`, `show_notification`, `create_ui_element`, `run_javascript`) and executes them live on the webpage via **LiteRT.js** and **WebGPU**.

### Why Gemma 3 270M?
- **Ultra-Compact (~150MB INT8 bundle)**: Downloads and compiles in WebGPU in under a second.
- **Lightning-Fast On-Device Inference**: Instantaneous token generation directly in client browsers with LiteRT.js.
- **Ungated Mirror**: `unsloth/gemma-3-270m-it` runs out of the box without requiring HF tokens.

## Step 1: Install Dependencies
Use `%pip` magic (works in Colab & Jupyter) or `!uv pip install` if using an Astral `uv` environment.

In [36]:
# In Google Colab or standard Jupyter kernel:
%pip install -q torch transformers peft trl datasets litert-torch litert-lm

# If using Astral uv locally in terminal / notebook:
# !uv pip install -q torch transformers peft trl datasets litert-torch litert-lm

/Users/xprilion/.local/share/uv/tools/jupyterlab/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## Step 2: Fine-Tune Gemma 3 270M as 'Snippy' with LoRA / PEFT

We load `unsloth/gemma-3-270m-it` and fine-tune it on Snippy's generic tool-calling dataset loaded directly from `snippy_dataset.json`.

In [37]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# Using Gemma 3 270M Instruction-Tuned model!
MODEL_ID = "unsloth/gemma-3-270m-it"
OUTPUT_LORA_DIR = "./lora_adapter"
OUTPUT_MERGED_DIR = "./fine_tuned_gemma_merged"

# 1. Load Tokenizer & Base Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# 2. Configure LoRA for Gemma 3
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
peft_model = get_peft_model(model, peft_config)

# 3. Load Snippy In-Browser Generic Tool Calling Dataset from snippy_dataset.json
with open('snippy_dataset.json', 'r') as f:
    sample_data = json.load(f)

dataset = Dataset.from_list(sample_data)
print(f"Loaded {len(dataset)} training examples from snippy_dataset.json")

# 4. Training with SFTTrainer
sft_config = SFTConfig(
    dataset_text_field="text",
    max_length=256,
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    logging_steps=1,
    loss_type="nll"
)
trainer = SFTTrainer(
    model=peft_model,
    train_dataset=dataset,
    args=sft_config,
)
trainer.train()

# Save LoRA Adapter
peft_model.save_pretrained(OUTPUT_LORA_DIR)
tokenizer.save_pretrained(OUTPUT_LORA_DIR)
print("✅ Snippy Gemma 3 270M LoRA Adapter Saved!")

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 2684.75it/s]


Loaded 14 training examples from snippy_dataset.json


Truncating train dataset: 100%|██████████| 14/14 [00:00<00:00, 8273.95 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 14/14 [00:00<00:00, 10818.03 examples/s]
/Users/xprilion/.local/share/uv/tools/jupyterlab/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,5.212310
2,5.334680
3,5.423196
4,5.246014
5,5.407609
6,6.046529
7,5.292041
8,5.078439
9,5.442470
10,4.682345


✅ Snippy Gemma 3 270M LoRA Adapter Saved!


## Step 3: Merge LoRA Adapter into Gemma 3 270M Base Model
Unload and merge Snippy's adapter weights into base weights.

In [38]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_ID = "unsloth/gemma-3-270m-it"
OUTPUT_LORA_DIR = "./lora_adapter"
OUTPUT_MERGED_DIR = "./fine_tuned_gemma_merged"

# Load base model & adapter, then merge
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32, device_map="cpu")
peft_model = PeftModel.from_pretrained(base_model, OUTPUT_LORA_DIR)
merged_model = peft_model.merge_and_unload()

# Save Merged Checkpoint
merged_model.save_pretrained(OUTPUT_MERGED_DIR)
tokenizer.save_pretrained(OUTPUT_MERGED_DIR)
print("✅ Merged Snippy Gemma 3 270M Model Saved to:", OUTPUT_MERGED_DIR)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.48it/s]

✅ Merged Snippy Gemma 3 270M Model Saved to: ./fine_tuned_gemma_merged


## Step 4: Convert Model to LiteRT Bundle (`.litertlm`)

Export model to LiteRT bundle format using `litert-torch export_hf` with `-b True`.

In [39]:
!litert-torch export_hf \
  ./fine_tuned_gemma_merged \
  ./litert_output \
  -b True \
  -q dynamic_int8

W0808 00:55:47.613000 9520 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W0808 00:55:47.627000 9520 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0808 00:55:48.559000 9520 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
============== Export Configuration ==============
aot_backend            : None
aot_compilation_config_dict : None
aot_soc_model          : None
assistant_model        : None
auto_model_override    : None
batch_size             : 1
bundle_litert_lm       : True
cache_implementation   : 'LiteRTLMCache'


## Step 5: Test Snippy with `litert-lm` Native Engine
Verify Snippy's generated JavaScript actions native response.

In [40]:
import litert_lm

# Initialize Engine with .litertlm bundle
engine = litert_lm.Engine('./litert_output/model.litertlm')
conv = engine.create_conversation()

res = conv.send_message('Snippy, change background color to dark purple.')
print("Snippy Response:", res['content'][0]['text'])

W0000 00:00:1786130783.224328  270264 litert_lm_loader.cc:291] Section not found: 
W0000 00:00:1786130783.224403  221652 litert_lm_loader.h:158] TFLite model type: TF_LITE_VISION_ENCODER not found for backend constraints. Skipping.
W0000 00:00:1786130783.224446  221652 litert_lm_loader.h:158] TFLite model type: TF_LITE_AUDIO_ENCODER_HW not found for backend constraints. Skipping.
W0000 00:00:1786130783.224454  221652 litert_lm_loader.h:174] TFLite model type: TF_LITE_VISION_ENCODER not found for prefer activation type. Use system's default backend activation type. System's default activation type for Text decoder is fp16. Vision encoder and audio encoder default is fp32.
W0000 00:00:1786130783.224457  221652 litert_lm_loader.h:174] TFLite model type: TF_LITE_AUDIO_ENCODER_HW not found for prefer activation type. Use system's default backend activation type. System's default activation type for Text decoder is fp16. Vision encoder and audio encoder default is fp32.
INFO: [environment.c

Snippy Response: Okay, I'm ready to help you with any background color changes you need! Please tell me what you want to change.



## Step 6: Deploy to Web Browser via LiteRT.js

Copy the exported model bundle and extract the TFLite FlatBuffer to the web server's static assets directory.

In [41]:
# Copy Converted Model Files to Local Web Demo Directory
import os
import shutil

SOURCE_MODEL = "./litert_output/model.litertlm"
DEST_DIR = "./web/public/models"

os.makedirs(DEST_DIR, exist_ok=True)
if os.path.exists(SOURCE_MODEL):
    # 1. Copy .litertlm bundle for Python backend engine
    shutil.copy(SOURCE_MODEL, os.path.join(DEST_DIR, "model.litertlm"))

    # 2. Extract TFLite FlatBuffer (starts with TFL3 magic) for WebGPU browser runtime
    with open(SOURCE_MODEL, "rb") as f:
        data = f.read()
    pos = data.find(b"TFL3")
    if pos != -1:
        tflite_data = data[pos - 4 :]
        with open(os.path.join(DEST_DIR, "model.tflite"), "wb") as f:
            f.write(tflite_data)
        print("✅ Extracted TFLite FlatBuffer and copied model files to web/public/models/")
    else:
        shutil.copy(SOURCE_MODEL, os.path.join(DEST_DIR, "model.tflite"))
        print("✅ Copied model files to web/public/models/")
else:
    print(f"⚠️ Source model not found at {SOURCE_MODEL}. Make sure Step 4 export finished.")

✅ Extracted TFLite FlatBuffer and copied model files to web/public/models/
